# Lab 4 — Data Quality Assessment & Preprocessing

**Course:** CYS 313: Computer Data Security and Privacy  
**Student:** HADI ABDULQADIER ALI ALMOAIREK  
**ID:** 2240002278  
**Section:** 6ms1  
**Date:** 4/2/2026

Dataset: `Chocolate_Sales.csv`


## Setup

In [ ]:
import pandas as pd
import numpy as np

csv_path = 'Chocolate_Sales.csv'
df = pd.read_csv(csv_path)
df.head()

## Task 1 — Identify data quality issues

In [ ]:
df.shape, df.dtypes

In [ ]:
df.isna().sum(), df.duplicated().sum()

In [ ]:
clean = df.copy()
clean['Date'] = pd.to_datetime(clean['Date'], dayfirst=True, errors='coerce')
clean['Amount'] = clean['Amount'].replace({'\\$':'', ',':''}, regex=True).astype(float)
clean['Boxes Shipped'] = pd.to_numeric(clean['Boxes Shipped'], errors='coerce')

{
  'missing_values': clean.isna().sum().to_dict(),
  'duplicates': int(clean.duplicated().sum()),
  'date_parse_nulls': int(clean['Date'].isna().sum()),
  'amount_minmax': (float(clean['Amount'].min()), float(clean['Amount'].max())),
  'boxes_minmax': (float(clean['Boxes Shipped'].min()), float(clean['Boxes Shipped'].max())),
}

## Task 2 — Missing value strategy
The original dataset has **no missing values**, so we create missing values artificially (5% of `Amount`) and apply **median imputation**.

**Why median?** It is more robust than mean when outliers exist.

In [ ]:
rng = np.random.default_rng(42)
miss = clean.copy()
mask = rng.random(len(miss)) < 0.05
miss.loc[mask, 'Amount'] = np.nan

missing_before = int(miss['Amount'].isna().sum())
median_amount = float(miss['Amount'].median())

imputed = miss.copy()
imputed['Amount'] = imputed['Amount'].fillna(median_amount)
missing_after = int(imputed['Amount'].isna().sum())

missing_before, median_amount, missing_after

## Task 3 — Outliers (IQR) and handling
We detect outliers in `Amount` with IQR and **cap** them to the IQR bounds.

In [ ]:
q1 = imputed['Amount'].quantile(0.25)
q3 = imputed['Amount'].quantile(0.75)
iqr = q3 - q1
lower = q1 - 1.5*iqr
upper = q3 + 1.5*iqr

outlier_count = int(((imputed['Amount'] < lower) | (imputed['Amount'] > upper)).sum())
out = imputed.copy()
out['Amount_capped'] = out['Amount'].clip(lower, upper)

outlier_count, float(lower), float(upper)

## Task 4 — Normalization (Min-Max and Z-score)
We scale `Amount_capped` and `Boxes Shipped`.

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

num = out[['Amount_capped','Boxes Shipped']].copy()
mm = MinMaxScaler()
zs = StandardScaler()

mm_arr = mm.fit_transform(num)
zs_arr = zs.fit_transform(num)

scaled = out.copy()
scaled['Amount_minmax'] = mm_arr[:,0]
scaled['Boxes_minmax']  = mm_arr[:,1]
scaled['Amount_z'] = zs_arr[:,0]
scaled['Boxes_z']  = zs_arr[:,1]

scaled[['Amount_capped','Amount_minmax','Amount_z','Boxes Shipped','Boxes_minmax','Boxes_z']].head()

## Task 5 — PCA
Apply PCA on standardized features and interpret explained variance.

In [ ]:
from sklearn.decomposition import PCA

X = scaled[['Amount_z','Boxes_z']].values
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X)

processed = scaled.copy()
processed['PC1'] = X_pca[:,0]
processed['PC2'] = X_pca[:,1]

pca.explained_variance_ratio_

### Interpretation
- PC1 explains **50.82%** of the variance.
- PC2 explains **49.18%** of the variance.
Because we only used two standardized numeric features, the variance is split almost evenly.

In [ ]:
processed.head()

In [ ]:
processed.to_csv('data_processed_with_pca.csv', index=False)
clean.to_csv('data_cleaned.csv', index=False)
imputed.to_csv('data_imputed.csv', index=False)
print('Saved outputs to CSV files.')